# Anchor-free detection: YOLO11s, input resolution and a P2 head (Kaggle or Colab, ~7 h T4)

Same protocol as the anchor-based detectors (RetinaNet, Faster R-CNN): train on **every 5th train frame**,
evaluate on **every 5th val frame** of the original val split, full frame, COCO AP with maxDets 500 broken
down by object size (small < 32², medium < 96², large), final-epoch weights (`last.pt`), score threshold 0.01.

| Parameter | Search range | Why it matters for anchor-free assignment |
|---|---|---|
| Input resolution (train = test) | **640** (Ultralytics default = initial model), 960, 1280 | object size in pixels -> grid points inside each object |
| Detection heads | **P3–P5** (default) vs **P2–P5** (extra stride-4 head, `configs/yolo11-p2.yaml`) at the best resolution; P2 at 640 if time is left | stride 4 puts 4x more points into each object |

Fixed for every run: `yolo11s.pt` (COCO-pretrained), 12 epochs, Ultralytics default optimizer, augmentation
and loss (CIoU + DFL box loss, TaskAlignedAssigner).

**Weights & Biases**: one W&B run per model (`yolo11s_<imgsz>`, `yolo11s_p2_<imgsz>`) with the full config,
train/val loss and mAP curves per epoch, learning rate, Ultralytics plots (PR curve, confusion matrix, val
batches), the final COCO AP by object size and 16 val frames with GT and predicted boxes; plus a
`yolo_anchor_free_summary` run with the assignment table, the comparison table and the AP-by-size chart.
Add your W&B API key as a secret named `WANDB_API_KEY` (Kaggle: **Add-ons -> Secrets**; Colab: key icon).
If the key cannot write to `WANDB_ENTITY`, the code retries with the key's default entity, then logs offline
(upload later with `wandb sync`), so training never stops because of W&B.

Runs on **Kaggle** (GPU T4, Internet on, add the competition data with **Add Input**, then
**Save & Run All**) or **Colab** (T4 runtime; Colab secret `KAGGLE_API_TOKEN`, or `KAGGLE_USERNAME` + `KAGGLE_KEY`, for the data
download; runs are kept on Google Drive, so after a disconnect **Run all** again: finished runs are skipped
and an interrupted one resumes from its `last.pt`).

## How YOLO assigns locations to objects without anchors

**Locations.** YOLO11 predicts on three feature maps P3/P4/P5 with strides 8/16/32. Every cell centre
((i + 0.5)·s, (j + 0.5)·s) is one *anchor point*: it predicts 8 class scores and, for each box side
(left, top, right, bottom), a 16-bin distribution (DFL) whose expectation is the distance from the point to
that side. No box shapes or sizes are predefined, so nothing has to be tuned to VisDrone's box shapes.
A 1920x1080 frame at `imgsz` 1280 is 1280x720 pixels: 160x90 + 80x45 + 40x23 ≈ 19k points.

**Label assignment** (Ultralytics `TaskAlignedAssigner`, topk = 10, α = 0.5, β = 6):
1. *Candidates*: the points whose centre lies **inside** the GT box, on any level.
2. *Alignment*: each candidate gets t = s^α · IoU^β, where s is its predicted score for the GT class and
   IoU is between its predicted box and the GT. The 10 candidates with the highest t become positives.
3. *Conflicts*: a point that is a positive for several GT boxes keeps only the GT its prediction overlaps most.
4. *Targets*: the class target is the normalised t (a soft label), the box is trained with CIoU + DFL.

Unlike anchor matching (a fixed IoU threshold between predefined boxes and the GT) the assignment is dynamic:
it follows the current predictions, and the FPN level is not fixed by object size.

**Small objects.** A w x w px object (after resizing) contains about (w / s)² points per level. VisDrone
frames are shrunk 3x at `imgsz` 640, so a 20 px pedestrian becomes ~7 px: it may contain **no** stride-8
centre at all, and then it never becomes a positive and is trained as background. Even with one or two
points the box must be regressed from a point near the object edge. A larger input or a stride-4 head
(P2) puts ~4x more points into every object; the table below counts this on the val annotations.

**Densely packed objects.** Neighbouring boxes overlap in parked cars and crowds, so points in the overlap are
candidates for several objects and each goes to only one of them. A small, occluded object with two points
can lose both. The topk = 10 cap keeps a large object from taking all points, but at inference NMS (IoU 0.7)
can still merge adjacent objects of the same class.

In [ ]:
import os, sys, time, re
from pathlib import Path

T0 = time.time()
ON_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ  # set only on Kaggle; Colab can also have /kaggle/input
WORK = Path("/kaggle/working" if ON_KAGGLE else "/content")
REPO = WORK / "repo"
if not ON_KAGGLE:
    from google.colab import drive
    drive.mount("/content/drive")
!nvidia-smi -L
![ -d {REPO}/.git ] && git -C {REPO} pull -q || git clone -q https://github.com/LiliAlania/drone-multi-object-detection.git {REPO}
%cd {REPO}
!pip install -q -U kaggle  # Colab's preinstalled kaggle CLI may not know the new API tokens
!pip install -q ultralytics==8.4.208 pycocotools wandb
sys.path.insert(0, str(REPO))
# the code comes from GitHub, not from this notebook: stop now if it predates this experiment
assert (REPO / "src/yolo_assign.py").is_file() and (REPO / "configs/yolo11-p2.yaml").is_file(), \
    "GitHub has an older version of the repo: commit and push before running this notebook"

In [ ]:
MODEL = "yolo11s.pt"
P2_MODEL = "configs/yolo11s-p2.yaml"  # the "s" picks the scale from configs/yolo11-p2.yaml
RESOLUTIONS = [640, 960, 1280]        # search range; 640 = Ultralytics default = the initial model
EPOCHS = 12                           # as RetinaNet in notebook 04
SUBSET_STEP = 5                       # every 5th train frame (~4.8k images), as the anchor-based runs
EVAL_STEP = 5                         # every 5th val frame
# Ultralytics accumulates gradients to a nominal batch of 64, so the batch size barely changes results;
# halve it if CUDA runs out of memory
BATCH = {640: 16, 960: 8, 1280: 8, 1536: 4}
WORKERS = 4 if ON_KAGGLE else 2
BUDGET_MIN = 690 if ON_KAGGLE else 11 * 60  # whole session
RESERVE_MIN = 30                            # kept free for the summary cells
RUNS = Path("/kaggle/working/yolo_runs") if ON_KAGGLE else Path("/content/drive/MyDrive/yolo_runs")
RUNS.mkdir(parents=True, exist_ok=True)

# W&B: a *team* entity you can write to (an organization name is refused); "" = the key's default entity
WANDB_PROJECT = "drone-multi-object-detection"
WANDB_ENTITY = ""
if WANDB_ENTITY:
    os.environ["WANDB_ENTITY"] = WANDB_ENTITY
os.environ["WANDB_DIR"] = str(RUNS)  # offline runs then land next to the checkpoints


def secret(name):
    try:
        if ON_KAGGLE:
            from kaggle_secrets import UserSecretsClient
            return UserSecretsClient().get_secret(name)
        from google.colab import userdata
        return userdata.get(name)
    except Exception as error:  # Colab: NotebookAccessError = the secret's "Notebook access" toggle is off
        if type(error).__name__ != "SecretNotFoundError":
            print(f"cannot read secret {name}: {type(error).__name__}: {error}")
        return None


# the `python -m` subprocesses inherit these; the kaggle CLI (Colab data download) reads the KAGGLE_* ones:
# KAGGLE_API_TOKEN (Kaggle -> Settings -> API -> Create New Token) or KAGGLE_USERNAME + KAGGLE_KEY (kaggle.json)
for name in ["WANDB_API_KEY"] + ([] if ON_KAGGLE else ["KAGGLE_API_TOKEN", "KAGGLE_USERNAME", "KAGGLE_KEY"]):
    value = secret(name)
    if value:
        os.environ[name] = value
if os.environ.get("KAGGLE_API_TOKEN"):  # also as the token file the CLI docs describe
    token_file = Path.home() / ".kaggle" / "access_token"
    token_file.parent.mkdir(exist_ok=True)
    token_file.write_text(os.environ["KAGGLE_API_TOKEN"])
    token_file.chmod(0o600)
if os.environ.get("KAGGLE_USERNAME") and os.environ.get("KAGGLE_KEY"):  # the new CLI reads the legacy key only from kaggle.json
    import json
    legacy_file = Path.home() / ".kaggle" / "kaggle.json"
    legacy_file.parent.mkdir(exist_ok=True)
    legacy_file.write_text(json.dumps({"username": os.environ["KAGGLE_USERNAME"], "key": os.environ["KAGGLE_KEY"]}))
    legacy_file.chmod(0o600)
HAVE_KAGGLE_KEY = ON_KAGGLE or bool(os.environ.get("KAGGLE_API_TOKEN") or os.environ.get("KAGGLE_KEY"))
if not ON_KAGGLE:  # which Kaggle login the CLI gets (never the secret itself)
    print("Kaggle credentials:", ", ".join(f"{k} ({len(os.environ[k])} chars)" for k in
          ["KAGGLE_API_TOKEN", "KAGGLE_USERNAME", "KAGGLE_KEY"] if os.environ.get(k)) or "none")
if not HAVE_KAGGLE_KEY:
    print("no Kaggle credentials: add the Colab secret KAGGLE_API_TOKEN (or KAGGLE_USERNAME + KAGGLE_KEY) "
          "with Notebook access on, then rerun this cell")
if "WANDB_API_KEY" not in os.environ:
    print("no WANDB_API_KEY secret: W&B logs offline")
if "WANDB_API_KEY" in os.environ:
    os.environ.pop("WANDB_MODE", None)  # an earlier run of this cell may have switched W&B offline
else:
    os.environ["WANDB_MODE"] = "offline"
print("platform:", "Kaggle" if ON_KAGGLE else "Colab", "| W&B:", os.environ.get("WANDB_MODE", "online"))

In [ ]:
# competition data: Kaggle input, or downloaded once per Colab session to local disk
INPUT = Path("/kaggle/input") if ON_KAGGLE else Path("/content/data")
find = lambda: [p for depth in range(0, 5) for p in INPUT.glob("*/" * depth + "val/annotations_coco.json")]
if not find() and not ON_KAGGLE:
    assert HAVE_KAGGLE_KEY, "add the Kaggle secret(s) in Colab (see the config cell output) and rerun the config cell"
    !python -m src.get_data --output-dir {INPUT}
found = find()
assert found, "Competition data not found: add it with Add Input (Kaggle) or set the Kaggle secrets (Colab)"
VAL_ROOT = found[0].parent
DATA = VAL_ROOT.parent
print("data:", DATA, sorted(p.name for p in DATA.iterdir()))

SUBSET = WORK / "visdrone-subset"
if not (SUBSET / "images" / "train").is_dir():
    !python -m src.make_subset --data-root {DATA} --output-dir {SUBSET} --step {SUBSET_STEP} --splits train val
    !rm -f {SUBSET}.zip
print(f"{len(list((SUBSET / 'images' / 'train').rglob('*.jpg')))} train images, "
      f"{(time.time() - T0) / 60:.0f} min elapsed")

### Grid points per object on the val annotations

For every val object (every 5th frame, ignored regions dropped) and every input size: the share of objects
whose resized box contains **no** grid-cell centre (never a positive), the median number of candidate points,
and the share of objects with a candidate point that also lies inside another object (dense scenes).
Nominal letterbox scale only: mosaic and scale augmentation change it during training.

In [ ]:
import pandas as pd

!python -m src.yolo_assign --split-root {VAL_ROOT} --step {EVAL_STEP} --imgsz 640 960 1280 1536 --out {RUNS}/assignment.csv
assign = pd.read_csv(RUNS / "assignment.csv")
assign.pivot_table(index=["imgsz", "heads"], columns="size",
                   values=["no_point_%", "median_points", "shared_point_%"]).round(1)

## Training and evaluation

Stage 1 trains YOLO11s at each resolution. Stage 2 adds the P2 head at the best resolution (by val mAP),
then at 640 if time is left: P2 at 640 has the same grid density as P3–P5 at 1280 at a quarter of the pixels.
Each run is evaluated right after training (`src.models.yolo_baseline evaluate`: full frame at the
training size, the same COCO code as Faster R-CNN), so a timeout loses at most the run in progress.

In [ ]:
METRIC = re.compile(r"mAP@\[\.5:\.95\] ([\d.]+) \| AP50 ([\d.]+) \| AP_small ([\d.]+) \| "
                    r"AP_medium ([\d.]+) \| AP_large ([\d.]+)")
run_minutes = []  # wall time of the runs trained in this session


def metrics(name):
    path = RUNS / name / "eval.txt"
    text = path.read_text(errors="ignore") if path.is_file() else ""
    ap, ms = METRIC.search(text), re.search(r"Inference: ([\d.]+) ms/image", text)
    if not ap:
        return None
    return {**dict(zip(["mAP", "AP50", "AP_small", "AP_medium", "AP_large"], map(float, ap.groups()))),
            "ms_per_image": float(ms[1]) if ms else None}


def run(name, imgsz, model=MODEL, batch=None):
    # get_ipython().system is what `!` runs; called directly so it works inside a function
    if metrics(name):
        print(f"{name}: trained and evaluated in an earlier session")
        return
    elapsed = (time.time() - T0) / 60
    if run_minutes and elapsed + max(run_minutes) > BUDGET_MIN - RESERVE_MIN:
        print(f"skip {name}: {elapsed:.0f} min used, a run takes up to ~{max(run_minutes):.0f} min")
        return
    out, start = RUNS / name, time.time()
    last = out / "weights" / "last.pt"
    resume = "--resume" if last.is_file() else ""  # after a disconnect, continue the interrupted run
    pretrained = f"--pretrained {MODEL}" if model.endswith(".yaml") else ""
    print(f"=== {name}: {model}, imgsz {imgsz} ===")
    get_ipython().system(
        f"python -m src.models.yolo_baseline train --data-root {SUBSET} --model {model} {pretrained} "
        f"--epochs {EPOCHS} --imgsz {imgsz} --batch {batch or BATCH[imgsz]} --workers {WORKERS} "
        f"--project {RUNS} --name {name} {resume} --wandb-project {WANDB_PROJECT}")
    get_ipython().system(
        f"python -m src.models.yolo_baseline evaluate --weights {last} --split-root {VAL_ROOT} "
        f"--step {EVAL_STEP} --imgsz {imgsz} --wandb-project {WANDB_PROJECT} | tee {out}/eval.txt")
    run_minutes.append((time.time() - start) / 60)
    print(f"{name}: {run_minutes[-1]:.0f} min, {(time.time() - T0) / 60:.0f} min elapsed")


# stage 1: input resolution
for imgsz in RESOLUTIONS:
    run(f"yolo11s_{imgsz}", imgsz)

In [ ]:
# stage 2: extra P2 head at the best resolution, then at 640
done = {imgsz: metrics(f"yolo11s_{imgsz}") for imgsz in RESOLUTIONS}
done = {imgsz: m for imgsz, m in done.items() if m}
assert done, "no resolution run finished: see the logs above"
BEST_IMGSZ = max(done, key=lambda imgsz: done[imgsz]["mAP"])
print("best resolution by val mAP:", BEST_IMGSZ)
for imgsz in dict.fromkeys([BEST_IMGSZ, 640]):  # P2 needs ~1.5x the memory: half the batch
    run(f"yolo11s_p2_{imgsz}", imgsz, model=P2_MODEL, batch=max(BATCH[imgsz] // 2, 1))

## Results: search range, selected settings, validation metrics

In [ ]:
rows = []
for name in sorted(p.name for p in RUNS.iterdir() if re.fullmatch(r"yolo11s_(p2_)?\d+", p.name)):
    m = metrics(name)
    if m:
        imgsz = int(name.rsplit("_", 1)[1])
        rows.append({"detector": "YOLO11s", "run": name, "imgsz": imgsz,
                     "heads": "P2-P5" if "_p2_" in name else "P3-P5", **m})
yolo = pd.DataFrame(rows).sort_values("mAP", ascending=False)
INITIAL = "yolo11s_640"
TUNED = yolo.iloc[0]["run"]
yolo["role"] = yolo["run"].map(lambda r: "initial = tuned" if r == INITIAL == TUNED else "initial" if r == INITIAL
                                else "tuned" if r == TUNED else "search")
yolo.to_csv(RUNS / "yolo_experiments.csv", index=False)
print(f"search range: imgsz {RESOLUTIONS} with P3-P5; P2-P5 at {sorted(set(yolo[yolo.heads == 'P2-P5'].imgsz))}")
print(f"initial: {INITIAL} | selected: {TUNED} (highest val mAP)")
print(f"settings: {EPOCHS} epochs, train every {SUBSET_STEP}th frame, val every {EVAL_STEP}th frame, last.pt")
yolo.round(4)

In [ ]:
from IPython.display import Image, display

display(Image(filename=str(RUNS / TUNED / "results.png")))  # Ultralytics training curves of the selected run

## Initial vs tuned YOLO vs tuned anchor-based detectors, by object size

The anchor-based numbers come from `results/anchor_based_reference.csv` (with their training frames and
epochs; the Faster R-CNN runs there used every 20th frame and 5–6 epochs). Replace them with the new
Faster R-CNN run of notebook 02 when it is done.

In [ ]:
import matplotlib.pyplot as plt

reference = pd.read_csv(REPO / "results" / "anchor_based_reference.csv")
ours = yolo[yolo.role != "search"].assign(train_frames=f"every {SUBSET_STEP}th", epochs=EPOCHS, source="this notebook")
ours["run"] = ours["run"] + " (" + ours["role"] + ")"
cols = ["detector", "run", "role", "mAP", "AP50", "AP_small", "AP_medium", "AP_large", "ms_per_image", "train_frames", "epochs"]
compare = pd.concat([ours, reference[reference.role == "tuned"]], ignore_index=True)[cols]
compare.to_csv(RUNS / "anchor_free_vs_anchor_based.csv", index=False)
display(compare.round(4))

sizes = ["AP_small", "AP_medium", "AP_large"]
labels = compare["detector"] + "\n" + compare["run"]
ax = compare[sizes].set_index(labels).plot.bar(figsize=(12, 4.5), rot=0, width=0.8)
ax.set_ylabel("COCO AP@[.5:.95]")
ax.set_title("AP by object size on every 5th val frame")
ax.tick_params(axis="x", labelsize=8)
plt.tight_layout()
plt.savefig(RUNS / "ap_by_size.png", dpi=150)
plt.show()

# one W&B run with everything needed for the report; a W&B problem must not hide the tables above
try:
    import wandb
    from src.models.faster_rcnn import wandb_run

    run = wandb_run(WANDB_PROJECT, RUNS / "yolo_anchor_free_summary", new=True, config={
        "resolutions": RESOLUTIONS, "best_imgsz": int(BEST_IMGSZ), "initial": INITIAL, "tuned": TUNED,
        "epochs": EPOCHS, "subset_step": SUBSET_STEP, "eval_step": EVAL_STEP})
    run.log({"yolo_experiments": wandb.Table(dataframe=yolo), "comparison": wandb.Table(dataframe=compare),
             "assignment": wandb.Table(dataframe=assign), "ap_by_size": wandb.Image(str(RUNS / "ap_by_size.png"))})
    run.finish()
except Exception as error:
    print("W&B summary not logged:", error)
print(f"{(time.time() - T0) / 60:.0f} min elapsed")

### What to read from the tables

- **Assignment vs AP_small**: compare the `no_point_%` of small objects at each resolution and head set with
  the AP_small of the matching run; resolution and P2 should help small objects most, large ones least.
- **P2 at 640 vs P3–P5 at 1280**: the same grid density, so a difference between them comes from the pixels
  the backbone sees (and the untrained P2 layers), not from the assignment.
- **Speed**: `ms_per_image` grows with imgsz² and with the P2 head; report it next to AP.
- **Fairness**: the anchor-based rows used other epochs / frame steps; say so when comparing them.

In [ ]:
# Kaggle output: keep only runs (weights, curves, eval logs, CSVs, plots)
if ON_KAGGLE:
    !rm -rf {SUBSET} {REPO}